# Communicating the model predictions

Four framings of the same out-of-sample NBA prediction data (logistic / GBT / production 0.75-GBT ensemble / market):
1. **Cumulative-accuracy race** — who pulls ahead over the season.
2. **Rolling divergence** — when the two learners disagree.
3. **Prediction distribution** — *why* GBT wins (it's sharper).
4. **Interactive explorer** (plotly) — hover per game.

Mirrors `pipelines/train.py`; predictions are out-of-sample (fit earlier, shown later).

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from train_eval_duckdb import load_events, build_market_pit, _pipeline, K, HFA, DEFAULT_DB
from sportsball.pipelines._elo import walk_forward
from sportsball.quant import features as feat
from sklearn.ensemble import HistGradientBoostingClassifier
GBT_W = 0.75   # production ensemble weight (strategy.ensemble_gbt_weight)

In [ ]:
raw = load_events(str(DEFAULT_DB))
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_,_,_) in raw]
frows,_ = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=90,
                       form_window=10, market_pit=build_market_pit(raw))
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
cut = int(0.85*len(X))
logit = _pipeline().fit(X[:cut], y[:cut])
gbt = HistGradientBoostingClassifier(max_iter=200,learning_rate=0.05,max_depth=3,random_state=0).fit(X[:cut],y[:cut])
pl, pg = logit.predict_proba(X[cut:])[:,1], gbt.predict_proba(X[cut:])[:,1]
mk = feat.FEATURE_ORDER.index('market_logit'); ml = X[cut:,mk]
g = pd.DataFrame({
  'date': pd.to_datetime([r[0] for r in raw[cut:]], errors='coerce'),
  'home':[r[1] for r in raw[cut:]], 'away':[r[2] for r in raw[cut:]],
  'P_logistic':pl, 'P_gbt':pg, 'P_ensemble':(1-GBT_W)*pl+GBT_W*pg,
  'P_market':np.where(ml!=0, 1/(1+np.exp(-ml)), np.nan), 'actual':y[cut:]})
g['label']=g.away+' @ '+g.home+'  '+g.date.dt.strftime('%Y-%m-%d')
g = g.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
print(len(g),'test games,', g.date.min().date(),'->',g.date.max().date())

## 1. Cumulative-accuracy race

Running accuracy as games accrue — the lines separate as the stronger model pulls ahead.

In [ ]:
def running_acc(prob, actual):
    valid = ~np.isnan(prob)
    correct = np.where(valid, ((prob>=0.5).astype(float)==actual).astype(float), 0.0)
    return np.cumsum(correct)/np.maximum(np.cumsum(valid.astype(float)),1)
fig, ax = plt.subplots(figsize=(11,5))
for col,lab,c in [('P_market','market','gray'),('P_logistic','logistic','steelblue'),
                  ('P_gbt','GBT','darkorange'),('P_ensemble','ensemble 0.75','crimson')]:
    ax.plot(g.index, running_acc(g[col].to_numpy(), g.actual.to_numpy()), label=lab, color=c, lw=1.5)
ax.set_xlim(200, len(g)); ax.set_xlabel('games (chronological)'); ax.set_ylabel('cumulative accuracy')
ax.legend(); ax.set_title('Cumulative-accuracy race (test set)'); plt.show()

## 2. Rolling divergence

Left: rolling-mean |logistic − GBT| (how far apart their probabilities are). Right: rolling rate at which they pick *different sides*. They mostly agree; the bumps are the interesting games.

In [ ]:
win = 100
diff = (g.P_logistic - g.P_gbt).abs().rolling(win, min_periods=20).mean()
flip = ((g.P_logistic>=0.5)!=(g.P_gbt>=0.5)).astype(float).rolling(win, min_periods=20).mean()
fig, ax1 = plt.subplots(figsize=(11,5)); ax2 = ax1.twinx()
ax1.plot(g.index, diff, color='purple', label='|logistic−GBT| (rolling)')
ax2.plot(g.index, flip*100, color='darkorange', ls='--', label='pick-disagreement % (rolling)')
ax1.set_xlabel('games'); ax1.set_ylabel('mean |prob diff|', color='purple'); ax2.set_ylabel('% games picking different sides', color='darkorange')
ax1.set_title(f'Rolling divergence (window {win})')
l1,la1=ax1.get_legend_handles_labels(); l2,la2=ax2.get_legend_handles_labels(); ax1.legend(l1+l2, la1+la2, loc='upper left'); plt.show()

## 3. Prediction distribution

Why GBT wins: it pushes probabilities toward 0/1 (more decisive), while the logistic clusters near 0.5. A sharper, still-calibrated model scores better.

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
for col,lab,c in [('P_logistic','logistic','steelblue'),('P_gbt','GBT','darkorange'),('P_ensemble','ensemble 0.75','crimson')]:
    sns.kdeplot(g[col], label=lab, color=c, lw=1.8, ax=ax, clip=(0,1))
ax.axvline(0.5, color='k', lw=.7, ls=':'); ax.set_xlabel('predicted P(home win)'); ax.set_xlim(0,1)
ax.legend(); ax.set_title('Distribution of predicted probabilities'); plt.show()

## 4. Interactive explorer (plotly)

The per-game dot plot, interactive — hover for the probability, filter models via the legend. Last 40 games. (Re-run the notebook to see it live; stripped from the committed file.)

In [ ]:
import plotly.express as px, plotly.graph_objects as go
import plotly.io as _pio, os as _os
_pio.templates.default = 'plotly_dark' if _os.getenv('NB_DARK') else 'plotly'
from IPython.display import HTML, display
recent = g.tail(40)
long = recent.melt(id_vars=['label'], value_vars=['P_market','P_logistic','P_gbt','P_ensemble'],
                   var_name='model', value_name='prob')
fig = px.scatter(long, x='prob', y='label', color='model', symbol='model',
                 title='What each model predicts (last 40 games)', height=900)
fig.add_trace(go.Scatter(x=recent.actual, y=recent.label, mode='markers', name='actual',
                         marker=dict(symbol='star', size=13, color='black')))
fig.add_vline(x=0.5, line_dash='dash', line_color='gray')
fig.update_xaxes(range=[-0.05,1.05], title='P(home win)'); fig.update_yaxes(autorange='reversed')
# Embed as interactive HTML so it survives nbconvert -> static HTML (and works live in Jupyter).
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))